In [ ]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates.
# These documents are dense, full of legal terminology, and often hundreds of pages long.
# The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections).
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma.
# - Retriever: When someone asks, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model generates a concise, plain-language summary.
# - Chat Loop: The compliance team can continue asking questions interactively.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
import gradio as gr

# Load PDF
loader = PyPDFLoader("data_privacy_regulation.pdf")
pages = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

# Embed and store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = Chroma.from_documents(chunks, embeddings)

# Load LLM
hf_pipeline = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Build RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 3})
)

def answer_query(question):
    return qa_chain.run(question)

# Gradio interface
demo = gr.Interface(
    fn=answer_query,
    inputs=gr.Textbox(label="Ask a legal question"),
    outputs=gr.Textbox(label="Answer"),
    title="Legal Research Assistant"
)

demo.launch()